<a href="https://colab.research.google.com/github/mariajosemuskusl/Integracion-de-Datos-y-Prospectiva/blob/main/5_Integraci%C3%B3n_Multidimensional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Caso de Estudio**

Se quiere hacer una clasificación de personas con diabetes a partir de su índice de masa corporal. Para este proceso se tomará como base el método K-Medoids (Integración Dinámica). En este caso de estudio se busca determinar los perfiles de cada grupo de personas. Para este proceso se utilizaran un total de 5 grupos de riesgo (Muy Bajo, Bajo, Medio, Alto, Muy Alto) sobre la variable BMI. Las variables que nos permitirán describir los perfiles de riesgo son las variables: (Describir todas las variables que tenemos)

**0. Se cargan las librerías de trabajo**

In [1]:
import numpy as np
import pandas as pd

from google.colab import files
uploaded = files.upload()

Mounted at /content/drive


**1. Se procede con la carga de los datos de trabajo**

In [6]:
nxl='/content/3. Parcial - medical_attention_data.xlsx'
xls=pd.ExcelFile(nxl)
sheet_names=xls.sheet_names
print(sheet_names)

XDB=pd.DataFrame()

for sheet in sheet_names:
  print(sheet)
  df=pd.read_excel(nxl,sheet_name=sheet)
  XDB=pd.concat([XDB,df], ignore_index=True)

#Procedemos ofuscación de los datos - desordenado de los datos
XDB2=XDB.sample(frac=1, random_state=42).reset_index(drop=True)
display(XDB2.head())

['Sabaneta', 'Bello', 'Medellín', 'Envigado', 'Itagui', 'Caldas']
Sabaneta
Bello
Medellín
Envigado
Itagui
Caldas


,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch,Pregnancies
0,SAB-0275,179.0,112,35,69,47.300,0.319,45,0,Sabaneta,NaN
1,ITA-0622,NaN,108,10,62,17.383,2.151,39,0,Itagui,NaN
2,CAL-0529,45.0,107,37,643,23.835,0.080,21,0,Caldas,NaN
3,BEL-0407,181.0,97,39,768,32.350,2.410,51,0,Bello,NaN
4,MED-0467,134.0,75,6,539,39.013,1.244,39,1,Medellín,9.0


**2. Se procede con la integración de los datos en 5 niveles de riesgo para la variable BMI**

In [25]:
XD=np.array(XDB2.iloc[:,5])
XC=np.array(XD[0:5,])    #Cinco cluster o concentardores de información
nc=np.zeros((len(XD),1)) #Indica el cluster al que pertenece cada dato

for k in range(len(XD)): #Para todos los pacientes
  nc[k,]=np.argmin(np.abs(XC-XD[k,])) #Esto me dice el cluster del paciente
  nc2=int(nc[k,]) #Aquí actualizamos el cluster
  XC[nc2,]=(XC[nc2,]+XD[k,])/2 #El cluster se actualiza a medida que llegan los datos

#Se procede con el despliegue de los clusters resultantes
df1=pd.DataFrame(XC)
display(df1.head())

#Desplegamos los pacientes del cluster 0
filas=np.where(nc==0) [0] #Quienes quedaron en el cluster cero
XDB.iloc[filas,].head()
#Porcentaje de pacientes con diabetes
npd=len(np.where(XDB.iloc[filas,8]==1)[0])
npn=len(np.where(XDB.iloc[filas,8]==0)[0])
print("El porcentaje de pacientes con diabetes es:",npd/(npd+npn))
print("El porcentaje de pacientes sin diabetes es:",npn/(npd+npn))

/tmp/ipykernel_14567/3235410609.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nc2=int(nc[k,]) #Aquí actualizamos el cluster


,0
0,46.760228
1,19.892895
2,26.900649
3,33.316196
4,41.556963


El porcentaje de pacientes con diabetes es: 0.49178403755868544
El porcentaje de pacientes sin diabetes es: 0.5082159624413145


**3. Se procede a determinar los perfiles de los pacientes por cluster para todas sus variables**

In [30]:
XDB2=XDB.iloc[:,1:8]  #Creamos una base de datos secundaria solo con variables numéricas

for j in range(5):  #5 clusters
  filas=np.where(nc==j)[0] #Las filas en donde quedaron los datos
  df2=XDB2.iloc[filas,]
  df3=pd.DataFrame(np.mean(df2,axis=0))
  df3.columns=['Cluster'+str(j+1)]
  display(df3.T)


,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Cluster1,121.40638,77.754695,29.665493,426.440141,32.643586,1.289479,50.928404


,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Cluster2,121.148344,80.872768,31.011161,417.448661,32.98094,1.25618,52.08817


,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Cluster3,117.970052,79.515118,29.227324,420.861142,32.499754,1.291617,51.908175


,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Cluster4,120.135135,79.540659,30.956044,420.154945,32.814796,1.321525,50.767033


,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Cluster5,121.84384,80.215962,30.255869,414.701878,33.001549,1.260089,51.463615


**4. Se procede con el ingreso de un nuevo paciente**

In [39]:
np_bmi=float(input("Ingresar el BMI del paciente")) #Se le hace la pregunta al paciente
#Se evalua su nivel de riesgo
import numpy as np #Re-import numpy to ensure 'np' refers to the numpy model
XCo=np.sort(XC)
ncluster=np.argmin(np.abs((XCo-np_bmi)))
print("El paciente pertenece al cluster",ncluster)

Ingresar el BMI del paciente50
El paciente pertenece al cluster 4
